# Hybrid Search

This notebook demonstrates **hybrid search** — combining dense (semantic) and sparse (keyword) retrieval to get the best of both worlds.

**Prerequisites:** [similarity_search_essentials.ipynb](./similarity_search_essentials.ipynb) — covers dense vs sparse vectors in depth.

---

## Why Hybrid Search?

| Method | Strengths | Weaknesses |
|---|---|---|
| **Dense (semantic)** | Understands meaning, handles paraphrasing | Misses exact keywords, rare terms |
| **Sparse (BM25)** | Precise keyword matching, great for IDs/names | No semantic understanding |
| **Hybrid** | Best of both — semantics + keyword precision | Slightly more complex to implement |

**Example:** Query `"BERT paper 2018"` — dense search finds semantically related content, but BM25 pins down the exact year and term.

In [ ]:
%pip install --upgrade pip > /dev/null
%pip install rank_bm25 faiss-cpu langchain langchain-community langchain-huggingface sentence-transformers numpy pandas > /dev/null

In [ ]:
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

## Sample Data

We use the same pet / wild animal corpus as in `similarity_search_essentials.ipynb` so results are easy to interpret — we expect dense search to cluster by topic and BM25 to match keywords precisely.

In [ ]:
documents = [
    "Dogs are loyal and friendly pets that live alongside humans.",
    "Golden Retrievers are great family dogs known for their gentle temperament.",
    "Cats are independent and clean animals that make wonderful house pets.",
    "Persian cats have long coats and calm personalities, popular as indoor pets.",
    "Rabbits are social house pets that enjoy interaction and play.",
    "Guinea pigs are gentle rodents and great pets for children.",
    "Hamsters are small nocturnal rodents that are easy to care for as pets.",
    "Parrots are intelligent birds that mimic speech and bond with their owners.",
    "Fish tanks reduce stress; fish are low-maintenance aquatic pets.",
    "Domestic dogs were bred over thousands of years to work with humans.",
    "Lions are apex predators living in social groups called prides in African savannas.",
    "Tigers are solitary hunters and the largest wild cats, found across Asia.",
    "Elephants are the largest land animals with complex matriarchal social structures.",
    "Wolves are intelligent pack animals found across North America, Europe, and Asia.",
    "Grizzly bears are powerful omnivores that hibernate during winter in North America.",
    "Cheetahs are the fastest land animals, reaching 70 mph when hunting prey.",
    "Gorillas are the largest primates, sharing 98% of DNA with humans.",
    "Polar bears are adapted to Arctic environments and hunt seals on sea ice.",
    "Jaguars are solitary wild cats native to the Americas, strong swimmers and climbers.",
    "African elephants use infrasound to communicate across long distances on the savanna.",
]

print(f"{len(documents)} documents loaded")

## Dense Search with FAISS

Dense search converts documents and queries to embedding vectors and finds nearest neighbours by cosine similarity.

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

from langchain.schema import Document
langchain_docs = [Document(page_content=doc, metadata={"index": i}) for i, doc in enumerate(documents)]

vector_store = FAISS.from_documents(langchain_docs, embedding_model)
print("FAISS index built")

In [ ]:
def dense_search(query, k=5):
    results = vector_store.similarity_search_with_score(query, k=k)
    return [(doc.page_content, float(score)) for doc, score in results]

query = "large wild cat predator"
print(f"Dense search results for: '{query}'\n")
for text, score in dense_search(query):
    print(f"  [{score:.3f}] {text}")

## Sparse Search with BM25

BM25 (Best Match 25) is a classic keyword ranking algorithm that scores documents based on term frequency and inverse document frequency. It excels when the query contains specific terms that must appear in results.

In [ ]:
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)
print("BM25 index built")

In [ ]:
def sparse_search(query, k=5):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:k]
    return [(documents[i], float(scores[i])) for i in top_indices]

print(f"BM25 search results for: '{query}'\n")
for text, score in sparse_search(query):
    print(f"  [{score:.3f}] {text}")

## Side-by-Side Comparison

Let's compare dense vs sparse results on queries where they behave differently.

In [ ]:
test_queries = [
    "fast hunting animal",           # dense should handle paraphrasing (cheetah)
    "polar arctic ice",              # BM25 matches keywords precisely
    "companion animal bonding",      # semantic — no exact keyword match
]

for q in test_queries:
    print(f"Query: '{q}'")
    dense = dense_search(q, k=3)
    sparse = sparse_search(q, k=3)
    print(f"  Dense  → {dense[0][0][:60]}...")
    print(f"  Sparse → {sparse[0][0][:60]}...")
    print()

## Hybrid Search with Reciprocal Rank Fusion (RRF)

**Reciprocal Rank Fusion (RRF)** merges ranked lists from multiple retrieval systems without needing to normalise scores across systems (scores from BM25 and cosine similarity are on different scales).

The RRF score for a document $d$ is:

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

Where:
- $R$ = set of ranked lists (dense results, sparse results)
- $\text{rank}_r(d)$ = rank of document $d$ in list $r$ (1-based)
- $k$ = smoothing constant (typically 60), prevents top-ranked documents from dominating

A document appearing at rank 1 in both lists scores higher than one appearing at rank 1 in only one list.

In [ ]:
def reciprocal_rank_fusion(ranked_lists, k=60):
    """
    Fuse multiple ranked lists using RRF.
    ranked_lists: list of lists of document strings (ordered best-first)
    Returns: list of (document, rrf_score) sorted by score descending
    """
    scores = {}
    for ranked_list in ranked_lists:
        for rank, doc in enumerate(ranked_list, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [ ]:
def hybrid_search(query, k=5, rrf_k=60):
    dense_results = [text for text, _ in dense_search(query, k=k)]
    sparse_results = [text for text, _ in sparse_search(query, k=k)]
    fused = reciprocal_rank_fusion([dense_results, sparse_results], k=rrf_k)
    return fused[:k]

query = "large wild cat predator"
print(f"Hybrid search results for: '{query}'\n")
for text, score in hybrid_search(query):
    print(f"  [{score:.4f}] {text}")

## Full Comparison: Dense vs Sparse vs Hybrid

In [ ]:
def compare_search_methods(query, k=3):
    dense = [t for t, _ in dense_search(query, k=k)]
    sparse = [t for t, _ in sparse_search(query, k=k)]
    hybrid = [t for t, _ in hybrid_search(query, k=k)]

    max_len = max(len(dense), len(sparse), len(hybrid))
    rows = []
    for i in range(max_len):
        rows.append({
            "Rank": i + 1,
            "Dense": dense[i][:55] + "..." if i < len(dense) else "",
            "Sparse (BM25)": sparse[i][:55] + "..." if i < len(sparse) else "",
            "Hybrid (RRF)": hybrid[i][:55] + "..." if i < len(hybrid) else "",
        })

    df = pd.DataFrame(rows).set_index("Rank")
    print(f"\nQuery: '{query}'")
    return df

compare_search_methods("wild animal hunting")

In [ ]:
compare_search_methods("animal living with humans")

In [ ]:
compare_search_methods("polar arctic ice")

## Tuning: Adjusting Dense vs Sparse Weight

RRF treats all retrieval systems equally. For more control, you can use **weighted score combination** — but this requires normalising scores to the same scale first.

In [ ]:
def hybrid_search_weighted(query, k=5, dense_weight=0.7, sparse_weight=0.3):
    """Weighted hybrid search with min-max normalised scores."""
    dense_results = dense_search(query, k=len(documents))
    sparse_results = sparse_search(query, k=len(documents))

    def normalise(results):
        scores = [s for _, s in results]
        min_s, max_s = min(scores), max(scores)
        if max_s == min_s:
            return {t: 0.0 for t, _ in results}
        return {t: (s - min_s) / (max_s - min_s) for t, s in results}

    dense_norm = normalise(dense_results)
    sparse_norm = normalise(sparse_results)

    all_docs = set(dense_norm) | set(sparse_norm)
    combined = {
        doc: dense_weight * dense_norm.get(doc, 0.0) + sparse_weight * sparse_norm.get(doc, 0.0)
        for doc in all_docs
    }
    return sorted(combined.items(), key=lambda x: x[1], reverse=True)[:k]

query = "fast hunting animal"
print(f"Weighted hybrid (dense=0.7, sparse=0.3) for: '{query}'\n")
for text, score in hybrid_search_weighted(query):
    print(f"  [{score:.3f}] {text}")

## When to Use Each Approach

| Scenario | Recommended |
|---|---|
| General semantic questions | Dense |
| Exact keyword / ID / name lookup | Sparse (BM25) |
| Mixed queries (meaning + keywords) | Hybrid RRF |
| Domain with rare technical terms | Hybrid or Sparse |
| No score normalisation needed | RRF (rank-based, scale-free) |
| Need to tune dense/sparse balance | Weighted combination |

Most production RAG pipelines default to **hybrid + RRF** — it consistently outperforms either method alone and requires no hyperparameter tuning.

## Additional Resources

- [Reciprocal Rank Fusion (Cormack et al., 2009)](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)
- [BM25 — Okapi BM25 (Wikipedia)](https://en.wikipedia.org/wiki/Okapi_BM25)
- [Hybrid Search in Weaviate](https://weaviate.io/blog/hybrid-search-explained)
- [Hybrid Search in Qdrant](https://qdrant.tech/articles/hybrid-search/)
- [FAISS — Facebook AI Similarity Search](https://github.com/facebookresearch/faiss)